In [1]:
from __future__ import annotations

from enum import Enum

from debugpy.common.log import LEVELS

import identifier

In [2]:
import errno
import sys

import lmdb
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

# from scripts.regsetup import description

import record_pb2
from meta import Run, Dir, File

In [4]:
from db import Db
# reload(db)

db_name = 'test-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

mydb = Db.open(path, readonly=readonly, create=create)
env = mydb.env

In [5]:
env, env.flags(), env.path(), env.info()

(<Environment at 0x1d9a3bdbb40>,
 {'subdir': True,
  'readonly': False,
  'metasync': True,
  'sync': True,
  'map_async': False,
  'readahead': True,
  'writemap': False,
  'meminit': False,
  'lock': True},
 'test-db',
 {'map_addr': 0,
  'map_size': 4294967296,
  'last_pgno': 7,
  'last_txnid': 24,
  'max_readers': 126,
  'num_readers': 1})

In [ ]:
with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            if key[:2] == b'r:':
                print(key, value)

In [ ]:
reload(db)

In [ ]:
run_id = '0016'
run = Run(
    run_id,
    path='',
    description='run from jupyter for development',
    specification='bla',
    start_time=1.234,
    end_time=1.234,
    duration=1.234,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    status='initialized')

from db import Db
key = Db.mk_run_key(run_id)
value = Db.mk_run_rec(run)
# value = msg.SerializeToString()

key, value


In [ ]:
mydb

In [ ]:
mydb.put_run(run)

In [ ]:
rune = mydb.get_run('0016')
rune

In [ ]:
rune.extra.items()

In [ ]:
env.close()

In [ ]:
with env.begin(write=True) as txn:
    txn.put(key, value)

txn

In [ ]:
with env.begin(write=False) as txn:
    c = txn.cursor()
    c.first()

In [ ]:
txn
# txn.stat()

In [ ]:
# Hieronder: spelen met ID classes

In [ ]:
from typing import Optional
# from importlib import reload

from identifier import Id, CompositeId
# reload Id, SubId

# class Id:
#
#     def __init__(self, val: int = 0, sup: Optional["Id"] = None, ext_len_bytes: int = 2):
#         self.val = val
#         self.sup = sup
#         self.ext_len = ext_len_bytes
#         self.len = (sup.len + ext_len_bytes) if sup else ext_len_bytes
#
#     def to_bytes(self) -> bytes:
#         if self.sup:
#             return self.sup.to_bytes() + self.val.to_bytes(length=self.ext_len)
#         return self.val.to_bytes(length=2)
#
#     def to_hex(self) -> str:
#         return self.to_bytes().hex()


In [7]:
# from importlib import reload

from identifier import Id, CompositeId
# reload Id, SubId


In [13]:
value = 256 + 13
idr = Id(value)
value

269

In [14]:
repr(idr), str(idr), idr.to_bytes(), idr.to_hex()

('Id(269)', '269', b'\x01\r', '010d')

In [15]:
idd = CompositeId(idr)
repr(idd), str(idd), idd.to_bytes(), idd.to_hex()

('SubId(269, 2)', '(269, 2)', b'\x01\r\x00\x02', '010d0002')

In [ ]:
from uuid import uuid4

u = uuid4()
u.__class__, str(u.__class__.__name__), str(u.__class__.__module__)

In [ ]:
idd.to_bytes(), idd.to_hex()
str(idd)

In [ ]:
idf = CompositeId(idd)

In [ ]:
idf.to_bytes(), idf.to_hex()
str(idf)

In [ ]:
id = Id(3)
id.to_bytes()

In [ ]:
from enum import Enum

class SubId(Id):
    class Level(Enum):
        ONE = 1
        TWO = 2

    LEVEL_SIZES = {
        Level.ONE: {"num_bytes": 2, "max_value": 256**2 - 1},
        Level.TWO: {"num_bytes": 4, "max_value": 256**4 - 1},
    }

    _last_id_by_level = {
        Level.ONE: 0,
        Level.TWO: 0,
    }

    def __init__(self, sup: Id):
        self.sup = sup
        if isinstance(sup, CompositeId):
            assert sup.level == CompositeId.Level.ONE, "No unexpected nesting level"
            self.level = CompositeId.Level.TWO
        else:  # sup is of class: Id
            self.level = CompositeId.Level.ONE

        last_id = CompositeId._last_id_by_level[self.level]
        val = last_id + 1
        assert val <= self.LEVEL_SIZES[self.level]["MAX_value"], ValueError("Exceeds max value")
        super().__init__(val, num_bytes=CompositeId.LEVEL_SIZES[self.level]["num_bytes"])
        CompositeId._last_id_by_level[self.level] = val

    def to_bytes(self):
        return self.sup.to_bytes() + super().to_bytes()


In [ ]:
did = SubId(id)
did.val, did.level, did.to_bytes()
dir(did)

In [ ]:
SubId._last_id_by_level

In [ ]:
fid = SubId(did)
(fid.val, fid.level, SubId._last_id_by_level)

In [ ]:
fid.to_bytes()


In [ ]:
'Xyz' + str([3, 2])
'.'.join([str(c) for c in [5, 4]])

In [ ]:
t = tuple([42, 7])
'Oeps' + str(t)